# 特徴量エンジニアリングとLasso回帰

`feature_engineering_dataset.csv` を使い、特徴量エンジニアリングを行ったあとにLasso回帰で予測します。

第3回の `lasso_medv_analysis.ipynb` から最小限の変更で作成しています。

【Python セミナー 第4回】  
特徴量エンジニアリングで作成した説明変数を使い、Lasso回帰を実行する。

In [ ]:
# 必要なライブラリを読み込む。
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import PolynomialFeatures


In [ ]:
# 予測結果を評価し、実測値と予測値の散布図を描く関数を定義する。
def pred_plot(train_targets, train_predictions, test_targets, test_predictions, model_name, target_name):
    train_mse = mean_squared_error(train_targets, train_predictions)
    train_mae = mean_absolute_error(train_targets, train_predictions)
    train_r2 = r2_score(train_targets, train_predictions)
    train_rmse = np.sqrt(train_mse)

    test_mse = mean_squared_error(test_targets, test_predictions)
    test_mae = mean_absolute_error(test_targets, test_predictions)
    test_r2 = r2_score(test_targets, test_predictions)
    test_rmse = np.sqrt(test_mse)

    train_targets_array = np.asarray(train_targets).ravel()
    train_predictions_array = np.asarray(train_predictions).ravel()
    test_targets_array = np.asarray(test_targets).ravel()
    test_predictions_array = np.asarray(test_predictions).ravel()

    print(f'\n{model_name} Training Set Performance:')
    print(f'Samples: {len(train_predictions)} | RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f} | R2: {train_r2:.4f}')

    print(f'\n{model_name} Test Set Performance:')
    print(f'Samples: {len(test_predictions)} | RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f} | R2: {test_r2:.4f}')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

    ax1.scatter(train_targets_array, train_predictions_array, alpha=0.6, color='red', s=20)
    train_min = min(train_targets_array.min(), train_predictions_array.min())
    train_max = max(train_targets_array.max(), train_predictions_array.max())
    train_range = np.linspace(train_min, train_max)
    ax1.plot(train_range, train_range, 'k--', lw=2)
    ax1.set_xlabel(f'Actual {target_name}')
    ax1.set_ylabel(f'Predicted {target_name}')
    ax1.set_title(f'{model_name} Training Set (n={len(train_predictions)})\nRMSE: {train_rmse:.3f} | MAE: {train_mae:.3f} | R2: {train_r2:.3f}')
    ax1.grid(True, alpha=0.3)
    ax1.set_aspect('equal', adjustable='box')

    ax2.scatter(test_targets_array, test_predictions_array, alpha=0.6, color='blue', s=20)
    test_min = min(test_targets_array.min(), test_predictions_array.min())
    test_max = max(test_targets_array.max(), test_predictions_array.max())
    test_range = np.linspace(test_min, test_max)
    ax2.plot(test_range, test_range, 'k--', lw=2)
    ax2.set_xlabel(f'Actual {target_name}')
    ax2.set_ylabel(f'Predicted {target_name}')
    ax2.set_title(f'{model_name} Test Set (n={len(test_predictions)})\nRMSE: {test_rmse:.3f} | MAE: {test_mae:.3f} | R2: {test_r2:.3f}')
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal', adjustable='box')

    plt.tight_layout()
    plt.show()


In [ ]:
# Lasso回帰で得られた係数を可視化する関数。
def lasso_coefficients(model, X, model_name):
    coefficients = model.coef_
    feature_names = X.columns

    coefficient_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})
    coefficient_df['AbsCoefficient'] = coefficient_df['Coefficient'].abs()
    coefficient_df = coefficient_df.sort_values(by='AbsCoefficient', ascending=False)

    display(coefficient_df[['Feature', 'Coefficient']])

    selected_df = coefficient_df[coefficient_df['AbsCoefficient'] > 0]

    plt.figure(figsize=(10, max(5, selected_df.shape[0] * 0.3)))
    plt.barh(selected_df['Feature'], selected_df['Coefficient'])
    plt.xlabel('Coefficient')
    plt.ylabel('Feature')
    plt.title(f'Coefficients in {model_name}')
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3)
    plt.show()

    print(f'使用された特徴量数: {selected_df.shape[0]} / {X.shape[1]}')


In [ ]:
# CSV ファイルを読み込む。
df = pd.read_csv('feature_engineering_dataset.csv')

# データを確認する。
df


,x1,x2,x3,x4,y
0,-1.254599,-3.148671,-2.382943,1.727030,4.474510
1,4.507143,0.419009,-2.530212,2.966814,16.006309
2,2.319939,3.729458,4.062546,-2.495321,-1.629434
3,0.986585,2.322249,-2.504538,1.248741,1.787929
4,-3.439814,3.065611,-2.280503,0.717460,5.071840
...,...,...,...,...,...
995,-4.084179,1.569552,3.652958,-0.578930,6.807607
996,4.173136,4.566146,-3.427268,-1.655988,9.779621
997,-3.631814,-4.310420,-1.902121,-1.054277,5.926408
998,4.502374,-4.429453,-2.099545,0.299406,11.194125


In [ ]:
# モデル学習に使わない列を削除する。
# id や smiles などの識別子列がある場合は、ここに列名を追加する。
drop_list = []

df_clean = df.drop(columns=drop_list)
df_clean


,x1,x2,x3,x4,y
0,-1.254599,-3.148671,-2.382943,1.727030,4.474510
1,4.507143,0.419009,-2.530212,2.966814,16.006309
2,2.319939,3.729458,4.062546,-2.495321,-1.629434
3,0.986585,2.322249,-2.504538,1.248741,1.787929
4,-3.439814,3.065611,-2.280503,0.717460,5.071840
...,...,...,...,...,...
995,-4.084179,1.569552,3.652958,-0.578930,6.807607
996,4.173136,4.566146,-3.427268,-1.655988,9.779621
997,-3.631814,-4.310420,-1.902121,-1.054277,5.926408
998,4.502374,-4.429453,-2.099545,0.299406,11.194125


In [ ]:
# 予測したい列を目的変数として指定する。
# データセットの列名が異なる場合は、この1行を変更する。
target_candidates = ['lifespan', 'Lifespan', 'material_lifespan', 'Material_Lifespan', 'target', 'y']
target = next((column for column in target_candidates if column in df_clean.columns), df_clean.columns[-1])

print(f'目的変数: {target}')

y = df_clean[target]
x = df_clean.drop(columns=[target])

# 数値列とカテゴリ列に分ける。
numeric_columns = x.select_dtypes(include='number').columns
categorical_columns = x.select_dtypes(exclude='number').columns

# 欠損値を補完する。
# 数値列は中央値、カテゴリ列は最頻値で補完する。
x_numeric = x[numeric_columns].copy()
x_numeric = x_numeric.fillna(x_numeric.median())

x_categorical = x[categorical_columns].copy()
for column in x_categorical.columns:
    x_categorical[column] = x_categorical[column].fillna(x_categorical[column].mode()[0])

# 特徴量エンジニアリング。
# PolynomialFeatures により、元の数値特徴量に加えて2乗項と交互作用項を作る。
poly = PolynomialFeatures(degree=2, include_bias=False)
x_numeric_engineered = poly.fit_transform(x_numeric)
engineered_feature_names = poly.get_feature_names_out(numeric_columns)
x_numeric_engineered = pd.DataFrame(x_numeric_engineered, columns=engineered_feature_names, index=x.index)

# カテゴリ列がある場合は、ダミー変数に変換する。
x_categorical_encoded = pd.get_dummies(x_categorical, drop_first=True, dtype=float)

# 数値の特徴量エンジニアリング結果とカテゴリのダミー変数を結合する。
x = pd.concat([x_numeric_engineered, x_categorical_encoded], axis=1)

print(f'元の説明変数数: {numeric_columns.shape[0] + categorical_columns.shape[0]}')
print(f'特徴量エンジニアリング後の説明変数数: {x.shape[1]}')
x.head()


目的変数: y


ValueError: No objects to concatenate

In [ ]:
# データをトレーニングセットとテストセットに分割する。
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.20, shuffle=True, random_state=42)

# 目的変数 y_train を標準化する。
auto_y_train = (y_train - y_train.mean()) / y_train.std()

# 説明変数 X_train を標準化する。
X_train_mean = X_train.mean()
X_train_std = X_train.std().replace(0, 1)
auto_X_train = (X_train - X_train_mean) / X_train_std

# テストデータは、トレーニングデータの平均と標準偏差を使って標準化する。
auto_X_test = (X_test - X_train_mean) / X_train_std

print(f'目的変数:{target},説明変数:{x.shape[1]},データ数:{x.shape[0]}')
print(f'トレーニングセット:{X_train.shape[0]},テストセット:{X_test.shape[0]}')


In [ ]:
# Lasso回帰を使って回帰モデルを作る。
model_name = 'Lasso with Feature Engineering'

# LassoCV は複数の alpha を試し、交差検証で最も良い alpha を自動的に選ぶ。
model = LassoCV(alphas=100, cv=5, max_iter=100000, random_state=42)

# 標準化したトレーニングデータでモデルを学習する。
model.fit(auto_X_train, auto_y_train)

print(f'最適な alpha: {model.alpha_:.6f}')

# トレーニングデータに対する予測値を計算し、元のスケールに戻す。
autoscaled_pred_y_train = model.predict(auto_X_train)
pred_y_train = autoscaled_pred_y_train * y_train.std() + y_train.mean()
pred_y_train = pd.DataFrame(pred_y_train, index=auto_X_train.index, columns=['pred_y'])

# テストデータに対する予測値を計算し、元のスケールに戻す。
autoscaled_pred_y_test = model.predict(auto_X_test)
pred_y_test = autoscaled_pred_y_test * y_train.std() + y_train.mean()
pred_y_test = pd.DataFrame(pred_y_test, index=X_test.index, columns=['pred_y'])

# 評価指標とグラフを表示する。
pred_plot(y_train, pred_y_train, y_test, pred_y_test, model_name, target)

# Lasso回帰で得られた係数を表示する。
lasso_coefficients(model, auto_X_train, model_name)


In [ ]:
# 予測値の一部を表で確認する。
prediction_sample = pd.DataFrame(
    {
        f'actual_{target}': y_test,
        f'predicted_{target}': pred_y_test['pred_y'],
    }
)

prediction_sample.head(10)
